In [1]:

# ================================================================
# AEGIS CLOUD — COMPLETE SINGLE-CELL PROJECT
# Cloud Reliability, Incident Intelligence & Safe Self-Healing
# Python + FastAPI + Interactive HTML Dashboard
#
# SAFE: All infrastructure and recovery operations are simulated.
# No shell commands, real server changes, or cloud resources used.
# ================================================================

import sys, subprocess, importlib.util, threading, time, json
from datetime import datetime, timezone
from collections import deque
from uuid import uuid4

# 1. Install dependency if missing
if importlib.util.find_spec("fastapi") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "fastapi"
    ])
if importlib.util.find_spec("uvicorn") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "uvicorn[standard]"
    ])

from fastapi import FastAPI, HTTPException
from fastapi.responses import HTMLResponse
from pydantic import BaseModel, Field
import uvicorn

# 2. Configuration and in-memory state
app = FastAPI(
    title="AEGIS CLOUD",
    description="Explainable cloud reliability and guarded recovery",
    version="3.0.0"
)

LOCK = threading.RLock()
LIMIT = 300

SCENARIOS = {
    "healthy": dict(cpu=24, memory=38, disk=42, errors=0.2,
                    service="running", api="healthy"),
    "cpu_spike": dict(cpu=96, memory=65, disk=45, errors=2,
                      service="running", api="degraded"),
    "memory_pressure": dict(cpu=60, memory=96, disk=48, errors=3,
                            service="running", api="degraded"),
    "disk_pressure": dict(cpu=45, memory=55, disk=97, errors=8,
                          service="running", api="failing"),
    "api_failure": dict(cpu=35, memory=45, disk=45, errors=75,
                        service="running", api="failing"),
    "service_down": dict(cpu=5, memory=15, disk=45, errors=100,
                         service="stopped", api="offline")
}

ACTIONS = {
    "simulate_disk_cleanup",
    "simulate_memory_recovery",
    "simulate_cpu_recovery",
    "simulate_service_restart",
    "simulate_api_recovery"
}

def timestamp():
    return datetime.now(timezone.utc).isoformat()

def new_state():
    return {
        **SCENARIOS["healthy"],
        "last_action": "System initialized",
        "history": deque(maxlen=LIMIT),
        "incidents": deque(maxlen=LIMIT),
        "audit": deque(maxlen=LIMIT)
    }

STATE = new_state()

def metrics():
    with LOCK:
        return {
            "timestamp": timestamp(),
            "cpu": STATE["cpu"],
            "memory": STATE["memory"],
            "disk": STATE["disk"],
            "error_rate": STATE["errors"],
            "service_status": STATE["service"],
            "api_status": STATE["api"],
            "last_action": STATE["last_action"],
            "simulation": True
        }

def log_event(action, result, details=None):
    with LOCK:
        STATE["audit"].append({
            "event_id": str(uuid4())[:8],
            "timestamp": timestamp(),
            "action": action,
            "result": result,
            "details": details or {}
        })

def collect():
    item = metrics()
    with LOCK:
        STATE["history"].append(item)
    return item

# 3. Explainable anomaly detection and root-cause ranking
def analyze():
    m = metrics()
    findings = []

    def add(cause, score, evidence, recommendation, action, severity):
        findings.append({
            "cause": cause,
            "score": score,
            "evidence": evidence,
            "recommendation": recommendation,
            "action": action,
            "severity": severity
        })

    if m["service_status"] == "stopped":
        add(
            "Service stopped", 0.99,
            "The simulated service is stopped.",
            "Inspect logs and dependencies before restarting.",
            "simulate_service_restart", "critical"
        )

    if m["disk"] >= 95:
        add(
            "Critical disk pressure", 0.96,
            f"Disk utilization is {m['disk']}%, threshold is 95%.",
            "Inspect disk consumers and approved temporary data.",
            "simulate_disk_cleanup", "critical"
        )
    elif m["disk"] >= 85:
        add(
            "Elevated disk utilization", 0.72,
            f"Disk utilization is {m['disk']}%.",
            "Inspect storage use and configure log rotation.",
            "inspect_disk", "warning"
        )

    if m["memory"] >= 90:
        add(
            "Memory pressure", 0.91,
            f"Memory utilization is {m['memory']}%.",
            "Inspect memory consumers and resource limits.",
            "simulate_memory_recovery", "critical"
        )

    if m["cpu"] >= 90:
        add(
            "CPU saturation", 0.89,
            f"CPU utilization is {m['cpu']}%.",
            "Investigate workload and request volume.",
            "simulate_cpu_recovery", "critical"
        )

    if m["error_rate"] >= 50:
        add(
            "Severe API errors", 0.98,
            f"Simulated API error rate is {m['error_rate']}%.",
            "Inspect application logs and dependency health.",
            "simulate_api_recovery", "critical"
        )
    elif m["error_rate"] >= 5:
        add(
            "Elevated API errors", 0.78,
            f"Simulated API error rate is {m['error_rate']}%.",
            "Inspect application errors and timeouts.",
            "simulate_api_recovery", "warning"
        )

    findings.sort(key=lambda x: x["score"], reverse=True)

    if not findings:
        findings = [{
            "cause": "No major anomaly detected",
            "score": 0.99,
            "evidence": "All metrics are below configured thresholds.",
            "recommendation": "Continue monitoring.",
            "action": "none",
            "severity": "normal"
        }]

    incident = {
        "incident_id": "INC-" + uuid4().hex[:8].upper(),
        "timestamp": timestamp(),
        "severity": findings[0]["severity"],
        "status": "healthy" if findings[0]["severity"] == "normal" else "open",
        "primary_cause": findings[0]["cause"],
        "findings": findings,
        "metrics": m,
        "score_note": "Heuristic ranking score, not an ML probability."
    }

    with LOCK:
        STATE["incidents"].append(incident)

    return incident

# 4. Simulated recovery with explicit approval and allowlisting
def recover(action, approved):
    if action not in ACTIONS:
        log_event(action, "rejected", {"reason": "Not allowlisted"})
        raise HTTPException(status_code=400, detail="Action not allowlisted.")

    if not approved:
        log_event(action, "blocked", {"reason": "Approval required"})
        return {
            "success": False,
            "message": "Recovery blocked. Explicit approval is required."
        }

    with LOCK:
        before = metrics()

        # These are state changes in Python only—not real operations.
        if action == "simulate_disk_cleanup":
            if STATE["disk"] < 85:
                return {
                    "success": False,
                    "message": "Disk usage is below the recovery threshold."
                }
            STATE["disk"] = 65

        elif action == "simulate_memory_recovery":
            STATE["memory"] = 55

        elif action == "simulate_cpu_recovery":
            STATE["cpu"] = 40

        elif action == "simulate_service_restart":
            STATE["service"] = "running"
            STATE["api"] = "healthy"
            STATE["errors"] = 0.2

        elif action == "simulate_api_recovery":
            STATE["api"] = "healthy"
            STATE["errors"] = 0.2

        STATE["last_action"] = action
        after = metrics()

    verified = (
        after["service_status"] == "running"
        and after["api_status"] == "healthy"
        and after["error_rate"] < 5
    )

    result = {
        "success": verified,
        "action": action,
        "before": before,
        "after": after,
        "verification": (
            "Simulated health checks passed."
            if verified else "Recovery incomplete; investigate further."
        )
    }

    collect()
    log_event(action, "verified" if verified else "verification_failed", result)
    return result

# 5. API endpoints
class ScenarioRequest(BaseModel):
    scenario: str

class RecoveryRequest(BaseModel):
    action: str
    approved: bool = False

@app.get("/api")
def api_info():
    return {
        "name": "AEGIS CLOUD",
        "version": "3.0.0",
        "mode": "safe simulation",
        "documentation": "/docs"
    }

@app.get("/health")
def health():
    m = metrics()
    ok = m["service_status"] == "running" and m["api_status"] == "healthy"
    return {"status": "ok" if ok else "degraded", **m}

@app.get("/api/metrics")
def get_metrics():
    collect()
    return metrics()

@app.get("/api/history")
def get_history():
    with LOCK:
        return list(STATE["history"])

@app.get("/api/scenarios")
def scenarios():
    return {"scenarios": list(SCENARIOS)}

@app.post("/api/scenario")
def choose_scenario(req: ScenarioRequest):
    if req.scenario not in SCENARIOS:
        raise HTTPException(
            status_code=400,
            detail=f"Choose one of: {list(SCENARIOS)}"
        )

    with LOCK:
        STATE.update(SCENARIOS[req.scenario])
        STATE["last_action"] = "Scenario: " + req.scenario

    collect()
    log_event("scenario_change", "success", {"scenario": req.scenario})
    return analyze()

@app.get("/api/analyze")
def get_analysis():
    return analyze()

@app.post("/api/recover")
def recovery_endpoint(req: RecoveryRequest):
    return recover(req.action, req.approved)

@app.get("/api/incidents")
def get_incidents():
    with LOCK:
        return list(STATE["incidents"])[::-1]

@app.get("/api/audit")
def get_audit():
    with LOCK:
        return list(STATE["audit"])[::-1]

@app.post("/api/reset")
def reset():
    with LOCK:
        STATE.clear()
        STATE.update(new_state())
    collect()
    log_event("reset", "success")
    return {"message": "Simulation reset.", "metrics": metrics()}

# 6. Interactive dashboard, served directly by FastAPI
HTML = r"""
<!DOCTYPE html><html><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>AEGIS CLOUD</title>
<style>
*{box-sizing:border-box}body{margin:0;background:#0a1020;color:#e9efff;
font:15px/1.55 system-ui,Arial}main{max-width:1150px;margin:auto;padding:25px}
h1{font-size:42px;margin:5px 0}h2{margin-top:0}
.sub{color:#9baaca}.label{color:#70adff;letter-spacing:3px;font-size:12px}
.grid{display:grid;grid-template-columns:repeat(4,1fr);gap:12px}
.card,.panel{background:#121d33;border:1px solid #2b3a57;border-radius:14px;padding:17px;margin:12px 0}
.value{font-size:32px;font-weight:750}.muted{color:#9baaca}
button,select{background:#1c3154;color:#fff;border:1px solid #385783;
border-radius:8px;padding:10px 12px;cursor:pointer;margin:4px}
button:hover{filter:brightness(1.2)}.primary{background:#235db7}
input{accent-color:#60a5fa}.notice{background:#332b17;padding:12px;
border-left:3px solid #facc15;border-radius:7px;color:#ffe7a0}
pre{white-space:pre-wrap;overflow-wrap:anywhere;background:#080d18;padding:12px;border-radius:8px}
table{border-collapse:collapse;width:100%;font-size:13px}td,th{text-align:left;
padding:9px;border-bottom:1px solid #2b3a57;vertical-align:top}
@media(max-width:700px){.grid{grid-template-columns:repeat(2,1fr)}main{padding:13px}}
</style></head><body><main>
<div class="label">CLOUD RELIABILITY INTELLIGENCE</div>
<h1>🛡️ AEGIS CLOUD</h1>
<p class="sub">Incident detection · Evidence-backed diagnosis · Guarded recovery</p>
<div class="notice">SAFE SIMULATION: no real server, files, or cloud resources are modified.</div>
<div class="grid" id="metrics"></div>

<section class="panel"><h2>01 / Incident Lab</h2>
<p class="muted">Inject a simulated infrastructure failure.</p>
<select id="scenario">
<option value="healthy">Healthy baseline</option><option value="cpu_spike">CPU spike</option>
<option value="memory_pressure">Memory pressure</option><option value="disk_pressure">Disk pressure</option>
<option value="api_failure">API failure</option><option value="service_down">Service down</option>
</select><button class="primary" onclick="applyScenario()">Apply scenario</button>
<button onclick="resetSim()">Reset</button><div id="analysis"></div></section>

<section class="panel"><h2>02 / Guarded Recovery</h2>
<p class="muted">Every action is allowlisted and requires explicit approval.</p>
<select id="action"><option value="simulate_disk_cleanup">Simulate disk cleanup</option>
<option value="simulate_memory_recovery">Simulate memory recovery</option>
<option value="simulate_cpu_recovery">Simulate CPU recovery</option>
<option value="simulate_service_restart">Simulate service restart</option>
<option value="simulate_api_recovery">Simulate API recovery</option></select>
<label><input type="checkbox" id="approved"> I approve this simulated action</label>
<button class="primary" onclick="doRecovery()">Execute recovery</button>
<div id="recovery"></div></section>

<section class="panel"><h2>03 / Incident History</h2><div id="incidents"></div></section>
<section class="panel"><h2>04 / Audit Trail</h2><div id="audit"></div></section>
<p class="muted">API docs: <a href="/docs" style="color:#70adff">/docs</a> |
Health: <a href="/health" style="color:#70adff">/health</a></p>
</main>
<script>
async function api(path,opts={}) {
 const r=await fetch(path,{headers:{'Content-Type':'application/json'},...opts});
 const d=await r.json();if(!r.ok)throw Error(d.detail||'Request failed');return d;
}
function esc(s){return String(s??'').replace(/[&<>"']/g,c=>({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[c]));}
function card(label,val,color){val=Number(val)||0;return `<div class="card"><div class="muted">${label}</div><div class="value" style="color:${color}">${val.toFixed(1)}%</div><progress max="100" value="${Math.min(100,val)}" style="width:100%"></progress></div>`;}
async function refresh(){
 const m=await api('/api/metrics');
 document.getElementById('metrics').innerHTML=
 card('CPU',m.cpu,'#70adff')+card('Memory',m.memory,'#b49bff')+
 card('Disk',m.disk,m.disk>=85?'#ff7185':'#4dd4a0')+
 card('API errors',m.error_rate,m.error_rate>=5?'#ff7185':'#4dd4a0');
 await analysis();await history();
}
async function analysis(){
 const a=await api('/api/analyze');
 document.getElementById('analysis').innerHTML=
 `<h3>Primary suspected cause: ${esc(a.primary_cause)}</h3>
 <p>Severity: <b>${esc(a.severity.toUpperCase())}</b></p>`+
 a.findings.map(f=>`<div class="card"><b>${esc(f.cause)}</b> · ranking score ${Math.round(f.score*100)}%
 <p><b>Evidence:</b> ${esc(f.evidence)}</p><p><b>Recommendation:</b> ${esc(f.recommendation)}</p>
 <p class="muted">Proposed action: ${esc(f.action)}</p></div>`).join('');
}
async function applyScenario(){
 try{await api('/api/scenario',{method:'POST',body:JSON.stringify({scenario:document.getElementById('scenario').value})});
 document.getElementById('recovery').innerHTML='';await refresh();}
 catch(e){alert(e.message);}
}
async function doRecovery(){
 try{const d=await api('/api/recover',{method:'POST',body:JSON.stringify({
 action:document.getElementById('action').value,
 approved:document.getElementById('approved').checked})});
 document.getElementById('recovery').innerHTML='<pre>'+esc(JSON.stringify(d,null,2))+'</pre>';
 await refresh();}catch(e){alert(e.message);}
}
async function resetSim(){
 try{await api('/api/reset',{method:'POST'});document.getElementById('recovery').innerHTML='';await refresh();}
 catch(e){alert(e.message);}
}
function table(items,cols){
 if(!items.length)return '<p class="muted">No records yet.</p>';
 return '<div style="overflow:auto"><table><tr>'+cols.map(c=>'<th>'+esc(c)+'</th>').join('')+'</tr>'+
 items.slice(0,10).map(x=>'<tr>'+cols.map(c=>'<td>'+esc(typeof x[c]==='object'?JSON.stringify(x[c]):x[c])+'</td>').join('')+'</tr>').join('')+'</table></div>';
}
async function history(){
 const incidents=await api('/api/incidents'),audit=await api('/api/audit');
 document.getElementById('incidents').innerHTML=table(incidents,['incident_id','timestamp','severity','primary_cause','status']);
 document.getElementById('audit').innerHTML=table(audit,['timestamp','action','result','details']);
}
refresh().catch(e=>alert(e.message));
setInterval(()=>refresh().catch(()=>{}),15000);
</script></body></html>
"""

@app.get("/", response_class=HTMLResponse)
def dashboard():
    return HTMLResponse(HTML)

# 7. Seed initial history
collect()
log_event("startup", "success", {"mode": "safe_simulation"})

# 8. Start server in the background
def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

existing = any(
    t.name == "aegis_cloud_server" and t.is_alive()
    for t in threading.enumerate()
)

if not existing:
    server_thread = threading.Thread(
        target=run_server, name="aegis_cloud_server", daemon=True
    )
    server_thread.start()

# 9. Wait for readiness and print usage
import urllib.request
ready = False
for _ in range(40):
    try:
        with urllib.request.urlopen(
            "http://127.0.0.1:8000/health", timeout=1
        ) as response:
            if response.status == 200:
                ready = True
                break
    except Exception:
        time.sleep(0.5)

if not ready:
    raise RuntimeError(
        "Server did not start. Check whether port 8000 is already in use."
    )

print("=" * 65)
print("AEGIS CLOUD IS RUNNING — SAFE SIMULATION MODE")
print("=" * 65)
print("Dashboard:      http://127.0.0.1:8000/")
print("API docs:       http://127.0.0.1:8000/docs")
print("Health:         http://127.0.0.1:8000/health")
print("Metrics:        http://127.0.0.1:8000/api/metrics")
print("Analysis:       http://127.0.0.1:8000/api/analyze")
print("Scenarios:      http://127.0.0.1:8000/api/scenarios")
print()
print("Try POST /api/scenario in /docs with:")
print('{"scenario":"disk_pressure"}')
print()
print("Recovery requires approval. All recovery actions are simulated.")
print("No AWS account or cloud spending required.")
print("=" * 65)

try:
    from IPython.display import display, HTML as IPyHTML
    display(IPyHTML(
        "<h3>AEGIS CLOUD is ready.</h3>"
        "<p>Open <a href='http://127.0.0.1:8000/docs' target='_blank'>"
        "the interactive API documentation</a>.</p>"
        "<p>The dashboard is served at <code>/</code> where localhost "
        "ports are accessible.</p>"
    ))
except ImportError:
    pass


AEGIS CLOUD IS RUNNING — SAFE SIMULATION MODE
Dashboard:      http://127.0.0.1:8000/
API docs:       http://127.0.0.1:8000/docs
Health:         http://127.0.0.1:8000/health
Metrics:        http://127.0.0.1:8000/api/metrics
Analysis:       http://127.0.0.1:8000/api/analyze
Scenarios:      http://127.0.0.1:8000/api/scenarios

Try POST /api/scenario in /docs with:
{"scenario":"disk_pressure"}

Recovery requires approval. All recovery actions are simulated.
No AWS account or cloud spending required.
